# Этап 10 V1 — остаточный модельный резерв: oracle OOF coverage

## Исследовательский вопрос

Есть ли в фиксированном наборе из семи уже сохранённых OOF score series верхняя граница резерва для rescue заранее принятой Stage 3 blind-spot cohort сверх `GBDT_mean`?

Это **детерминированная диагностическая** проверка без обучения, новых prediction, tuning, blending, calibration и final test. `oracle-any` — объединение top-K семи рядов и только upper-bound diagnostic; он не является системой с общей capacity 30%, поскольку размер объединения может превышать K.

## Locked contract

Используются ровно семь score series: CatBoost, XGBoost и LightGBM Stage 3; `GBDT_mean` и TabM stacking V1 Stage 7; TabM standalone V4 Stage 6; FT-Transformer V1 Stage 8. RealMLP и любые другие модели исключены.

Ранжирование: probability по убыванию; точный tie — `working_index` по возрастанию. Проверяются K10=28 961, K20=57 922, K30=86 884. Любое нарушение identity/alignment contract вызывает `assert` и останавливает notebook.

Stage 6 V4 не содержит `working_indices`. Поэтому проверяются JSON `working_index_sha256`, точное равенство `y_true` каноническому Stage 3 target, shape `(289614,)` и конечность `oof_probability`; лишь затем применяется сохранённый порядок Stage 6.

In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'reports' / 'generated').is_dir():
            return candidate
    raise RuntimeError('Не найден корень проекта: проверены cwd и его parents.')


ROOT = find_project_root(Path.cwd())
GENERATED = ROOT / 'reports' / 'generated'
SUMMARY = ROOT / 'reports' / 'summary'
STAGE3_OOF = GENERATED / 'stage3_oof_predictions_V1.npz'
STAGE3_RESULTS = GENERATED / 'stage3_error_analysis_results_V1.json'
STAGE6_OOF = GENERATED / 'stage6_tabm_oof_V4.npz'
STAGE6_RESULTS = GENERATED / 'stage6_tabm_results_V4.json'
STAGE7_OOF = GENERATED / 'stage7_tabm_stacking_oof_V1.npz'
STAGE8_OOF = GENERATED / 'stage8_ft_transformer_oof_V1.npz'
STAGE9_RESULTS = GENERATED / 'stage9_rank_capacity_results_V1.json'

EXPECTED_N = 289_614
EXPECTED_WORKING_SHA256 = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
EXPECTED_BLIND_SPOT_N = 805
CAPACITIES = {10: 28_961, 20: 57_922, 30: 86_884}
LOCKED_SCORE_NAMES = (
    'CatBoost_Stage3', 'XGBoost_Stage3', 'LightGBM_Stage3', 'GBDT_mean_Stage7',
    'TabM_standalone_V4_Stage6', 'TabM_stacking_V1_Stage7', 'FT_Transformer_V1_Stage8',
)
BASELINE_NAME = 'GBDT_mean_Stage7'

print(f'Project ROOT: {ROOT}')
print('Locked score set:', ', '.join(LOCKED_SCORE_NAMES))

Project ROOT: D:\Projects\komus-work
Locked score set: CatBoost_Stage3, XGBoost_Stage3, LightGBM_Stage3, GBDT_mean_Stage7, TabM_standalone_V4_Stage6, TabM_stacking_V1_Stage7, FT_Transformer_V1_Stage8


In [2]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def sha256_array(values: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(values).tobytes()).hexdigest()


stage3_diagnostics = json.loads(STAGE3_RESULTS.read_text(encoding='utf-8'))
stage6_diagnostics = json.loads(STAGE6_RESULTS.read_text(encoding='utf-8'))
stage9_diagnostics = json.loads(STAGE9_RESULTS.read_text(encoding='utf-8'))
accepted_rules = stage3_diagnostics['диагностические_границы']
consensus_rank_lte = float(accepted_rules['глубоко_пропущенный_дефолт_consensus_rank_lte'])
rank_spread_lte = float(accepted_rules['общая_слепая_зона_rank_spread_lte'])
assert (consensus_rank_lte, rank_spread_lte) == (0.50, 0.10)

with np.load(STAGE3_OOF, allow_pickle=False) as stage3, \
     np.load(STAGE6_OOF, allow_pickle=False) as stage6, \
     np.load(STAGE7_OOF, allow_pickle=False) as stage7, \
     np.load(STAGE8_OOF, allow_pickle=False) as stage8:
    required_stage3 = {'working_indices', 'target', 'oof_catboost', 'oof_xgboost', 'oof_lightgbm', 'consensus_rank', 'rank_spread'}
    required_stage6 = {'y_true', 'oof_probability'}
    required_stage7 = {'working_indices', 'target', 'gbdt_mean', 'tabm_oof_probability'}
    required_stage8 = {'working_indices', 'target', 'ft_transformer_oof_probability'}
    assert required_stage3.issubset(stage3.files), 'Stage 3 artifact lacks required arrays.'
    assert required_stage6.issubset(stage6.files), 'Stage 6 V4 artifact lacks required arrays.'
    assert required_stage7.issubset(stage7.files), 'Stage 7 artifact lacks required arrays.'
    assert required_stage8.issubset(stage8.files), 'Stage 8 artifact lacks required arrays.'

    working_indices = np.asarray(stage3['working_indices'], dtype=np.int64)
    target = np.asarray(stage3['target'], dtype=np.int8)
    working_indices_7 = np.asarray(stage7['working_indices'], dtype=np.int64)
    target_7 = np.asarray(stage7['target'], dtype=np.int8)
    working_indices_8 = np.asarray(stage8['working_indices'], dtype=np.int64)
    target_8 = np.asarray(stage8['target'], dtype=np.int8)
    stage6_y_true = np.asarray(stage6['y_true'], dtype=np.int8)

    stage6_oof_probability = np.asarray(stage6['oof_probability'], dtype=np.float64)
    stage3_7_8_scores = {
        'CatBoost_Stage3': np.asarray(stage3['oof_catboost'], dtype=np.float64),
        'XGBoost_Stage3': np.asarray(stage3['oof_xgboost'], dtype=np.float64),
        'LightGBM_Stage3': np.asarray(stage3['oof_lightgbm'], dtype=np.float64),
        'GBDT_mean_Stage7': np.asarray(stage7['gbdt_mean'], dtype=np.float64),
        'TabM_stacking_V1_Stage7': np.asarray(stage7['tabm_oof_probability'], dtype=np.float64),
        'FT_Transformer_V1_Stage8': np.asarray(stage8['ft_transformer_oof_probability'], dtype=np.float64),
    }
    consensus_rank = np.asarray(stage3['consensus_rank'], dtype=np.float64)
    rank_spread = np.asarray(stage3['rank_spread'], dtype=np.float64)

assert working_indices.shape == target.shape == (EXPECTED_N,)
assert working_indices_7.shape == target_7.shape == (EXPECTED_N,)
assert working_indices_8.shape == target_8.shape == (EXPECTED_N,)
assert np.array_equal(working_indices, working_indices_7)
assert np.array_equal(working_indices, working_indices_8)
assert np.array_equal(target, target_7)
assert np.array_equal(target, target_8)
assert sha256_array(working_indices) == EXPECTED_WORKING_SHA256
# Stage 6 V4: no invented working_indices key; check its documented alignment contract first.
assert stage6_diagnostics['working_index_sha256'] == EXPECTED_WORKING_SHA256
assert stage6_y_true.shape == (EXPECTED_N,)
assert np.array_equal(stage6_y_true, target)
assert stage6_oof_probability.shape == (EXPECTED_N,)
assert np.isfinite(stage6_oof_probability).all()

# The saved Stage 6 order enters the locked score set only after its special contract passes.
scores = {
    'CatBoost_Stage3': stage3_7_8_scores['CatBoost_Stage3'],
    'XGBoost_Stage3': stage3_7_8_scores['XGBoost_Stage3'],
    'LightGBM_Stage3': stage3_7_8_scores['LightGBM_Stage3'],
    'GBDT_mean_Stage7': stage3_7_8_scores['GBDT_mean_Stage7'],
    'TabM_standalone_V4_Stage6': stage6_oof_probability,
    'TabM_stacking_V1_Stage7': stage3_7_8_scores['TabM_stacking_V1_Stage7'],
    'FT_Transformer_V1_Stage8': stage3_7_8_scores['FT_Transformer_V1_Stage8'],
}
assert tuple(scores) == LOCKED_SCORE_NAMES
assert all(values.shape == (EXPECTED_N,) and np.isfinite(values).all() for values in scores.values())

preflight = {
    'n': EXPECTED_N,
    'expected_working_index_sha256': EXPECTED_WORKING_SHA256,
    'stage3_stage7_stage8': {
        'working_indices_exact_equal': True,
        'target_exact_equal': True,
        'working_index_sha256_pass': True,
    },
    'stage6_v4_special_alignment': {
        'working_indices_in_npz': False,
        'results_working_index_sha256_pass': True,
        'y_true_exact_equal_canonical_stage3_target': True,
        'y_true_shape': list(stage6_y_true.shape),
        'oof_probability_shape': list(stage6_oof_probability.shape),
        'oof_probability_finite': True,
        'saved_order_used_only_after_checks': True,
    },
    'all_locked_probability_arrays_finite': True,
    'locked_score_series_count': len(scores),
    'accepted_stage3_thresholds': {
        'consensus_rank_lte': consensus_rank_lte,
        'rank_spread_lte': rank_spread_lte,
    },
}
display(pd.json_normalize(preflight, sep='.'))

,n,expected_working_index_sha256,all_locked_probability_arrays_finite,locked_score_series_count,stage3_stage7_stage8.working_indices_exact_equal,stage3_stage7_stage8.target_exact_equal,stage3_stage7_stage8.working_index_sha256_pass,stage6_v4_special_alignment.working_indices_in_npz,stage6_v4_special_alignment.results_working_index_sha256_pass,stage6_v4_special_alignment.y_true_exact_equal_canonical_stage3_target,stage6_v4_special_alignment.y_true_shape,stage6_v4_special_alignment.oof_probability_shape,stage6_v4_special_alignment.oof_probability_finite,stage6_v4_special_alignment.saved_order_used_only_after_checks,accepted_stage3_thresholds.consensus_rank_lte,accepted_stage3_thresholds.rank_spread_lte
0,289614,80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d...,True,7,True,True,True,False,True,True,[289614],[289614],True,True,0.5,0.1


In [3]:
# Reconstruct the accepted Stage 3/9 blind-spot cohort and verify its already fixed row-level identity.
blind_spot = (target == 1) & (consensus_rank <= consensus_rank_lte) & (rank_spread <= rank_spread_lte)
assert int(blind_spot.sum()) == EXPECTED_BLIND_SPOT_N
assert bool(np.all(target[blind_spot] == 1))
blind_spot_working_indices = np.asarray(working_indices[blind_spot], dtype=np.int64)
blind_spot_cohort_sha256 = sha256_array(blind_spot_working_indices)
assert stage9_diagnostics['blind_spot']['count'] == EXPECTED_BLIND_SPOT_N
assert stage9_diagnostics['blind_spot']['all_target_one'] is True
assert blind_spot_cohort_sha256 == stage9_diagnostics['blind_spot_cohort_sha256']

blind_spot_check = {
    'count': int(blind_spot.sum()),
    'all_target_one': bool(np.all(target[blind_spot] == 1)),
    'rule': 'target == 1 AND consensus_rank <= 0.50 AND rank_spread <= 0.10',
    'blind_spot_cohort_sha256': blind_spot_cohort_sha256,
    'stage9_row_level_identity_pass': True,
}
display(pd.DataFrame([blind_spot_check]))

,count,all_target_one,rule,blind_spot_cohort_sha256,stage9_row_level_identity_pass
0,805,True,target == 1 AND consensus_rank <= 0.50 AND ran...,f0c227d1f07b922300b5488eb11c48590f0527e871802f...,True


In [4]:
def exact_top_k(probability: np.ndarray, k: int) -> np.ndarray:
    """Probability DESC; exact tie -> working_index ASC."""
    order = np.lexsort((working_indices, -np.asarray(probability, dtype=np.float64)))
    selected = np.zeros(EXPECTED_N, dtype=bool)
    selected[order[:k]] = True
    assert int(selected.sum()) == k
    return selected


default_count = int(target.sum())
alternative_names = tuple(name for name in LOCKED_SCORE_NAMES if name != BASELINE_NAME)
capacity_records = []
display_rows = []

for capacity_pct, k in CAPACITIES.items():
    selected = {name: exact_top_k(score, k) for name, score in scores.items()}
    oracle_any = np.logical_or.reduce(tuple(selected.values()))
    oracle_union_size = int(oracle_any.sum())
    assert oracle_union_size >= k

    score_metrics = {
        name: {
            'blind_spot_rescue_count': int((blind_spot & mask).sum()),
            'blind_spot_rescue_rate': float((blind_spot & mask).sum() / EXPECTED_BLIND_SPOT_N),
            'overall_default_capture': float((target.astype(bool) & mask).sum() / default_count),
        }
        for name, mask in selected.items()
    }
    baseline = selected[BASELINE_NAME]
    gbdt_missed_blind_spot = blind_spot & ~baseline
    oracle_rescue_missed = gbdt_missed_blind_spot & oracle_any

    contributions = {}
    prior_alternatives = np.zeros(EXPECTED_N, dtype=bool)
    for name in alternative_names:
        direct = gbdt_missed_blind_spot & selected[name]
        incremental = direct & ~prior_alternatives
        other_alternatives = np.logical_or.reduce(tuple(selected[other] for other in alternative_names if other != name))
        unique = direct & ~other_alternatives
        contributions[name] = {
            'rescue_count_among_gbdt_mean_missed': int(direct.sum()),
            'rescue_rate_among_blind_spot': float(direct.sum() / EXPECTED_BLIND_SPOT_N),
            'unique_contribution_count': int(unique.sum()),
            'incremental_contribution_count_locked_order': int(incremental.sum()),
        }
        prior_alternatives |= selected[name]

    incremental_sum = sum(value['incremental_contribution_count_locked_order'] for value in contributions.values())
    assert incremental_sum == int(oracle_rescue_missed.sum())
    oracle_rescue_count = int((blind_spot & oracle_any).sum())
    baseline_rescue_count = score_metrics[BASELINE_NAME]['blind_spot_rescue_count']
    assert oracle_rescue_count == baseline_rescue_count + int(oracle_rescue_missed.sum())

    oracle_metrics = {
        'actual_union_size_U_c': oracle_union_size,
        'effective_union_capacity': float(oracle_union_size / EXPECTED_N),
        'excess_rows': int(oracle_union_size - k),
        'blind_spot_rescue_count': oracle_rescue_count,
        'blind_spot_rescue_rate': float(oracle_rescue_count / EXPECTED_BLIND_SPOT_N),
        'overall_default_capture': float((target.astype(bool) & oracle_any).sum() / default_count),
    }
    capacity_records.append({
        'capacity_pct': capacity_pct,
        'K': k,
        'score_series': score_metrics,
        'GBDT_mean': score_metrics[BASELINE_NAME],
        'oracle_any': oracle_metrics,
        'alternative_model_contributions_to_gbdt_mean_missed_blind_spot': contributions,
        'consistency': {
            'oracle_rescue_among_gbdt_mean_missed_count': int(oracle_rescue_missed.sum()),
            'incremental_contributions_sum': incremental_sum,
            'oracle_rescue_equals_gbdt_mean_rescue_plus_residual': True,
        },
    })
    display_rows.append({
        'Capacity': f'{capacity_pct}%', 'K': k, 'Oracle U_c': oracle_union_size,
        'Effective union capacity, %': 100 * oracle_metrics['effective_union_capacity'],
        'Excess rows': oracle_metrics['excess_rows'],
        'GBDT_mean rescue, n': baseline_rescue_count,
        'Oracle-any rescue, n': oracle_rescue_count,
        'Δ oracle rescue vs GBDT, p.p.': 100 * (oracle_metrics['blind_spot_rescue_rate'] - score_metrics[BASELINE_NAME]['blind_spot_rescue_rate']),
        'GBDT_mean default capture, %': 100 * score_metrics[BASELINE_NAME]['overall_default_capture'],
        'Oracle-any default capture, %': 100 * oracle_metrics['overall_default_capture'],
    })

capacity_table = pd.DataFrame(display_rows)
display(capacity_table.round(3))

,Capacity,K,Oracle U_c,"Effective union capacity, %",Excess rows,"GBDT_mean rescue, n","Oracle-any rescue, n","Δ oracle rescue vs GBDT, p.p.","GBDT_mean default capture, %","Oracle-any default capture, %"
0,10%,28961,39850,13.760,10889,0,0,0.000,57.416,67.264
1,20%,57922,74603,25.759,16681,0,0,0.000,78.044,84.337
2,30%,86884,108450,37.446,21566,0,15,1.863,87.475,91.676


In [5]:
record_30 = next(record for record in capacity_records if record['capacity_pct'] == 30)
primary_delta_oracle_blind_spot_rescue_30_pp = 100 * (
    record_30['oracle_any']['blind_spot_rescue_rate'] - record_30['GBDT_mean']['blind_spot_rescue_rate']
)
decision = (
    'oracle_reserve_present'
    if primary_delta_oracle_blind_spot_rescue_30_pp >= 5.0
    else 'limited_residual_model_reserve'
)

contribution_table = pd.DataFrame.from_dict(
    record_30['alternative_model_contributions_to_gbdt_mean_missed_blind_spot'], orient='index'
).rename_axis('alternative_model').reset_index()
display(contribution_table)
print(f'Primary Δ oracle blind-spot rescue @30: {primary_delta_oracle_blind_spot_rescue_30_pp:.3f} п.п.')
print(f'Decision: {decision}')
print('Внимание: oracle-any — upper bound; его U_c не равен общей fixed capacity K.')

,alternative_model,rescue_count_among_gbdt_mean_missed,rescue_rate_among_blind_spot,unique_contribution_count,incremental_contribution_count_locked_order
0,CatBoost_Stage3,0,0.000000,0,0
1,XGBoost_Stage3,0,0.000000,0,0
2,LightGBM_Stage3,0,0.000000,0,0
3,TabM_standalone_V4_Stage6,6,0.007453,6,6
4,TabM_stacking_V1_Stage7,0,0.000000,0,0
5,FT_Transformer_V1_Stage8,9,0.011180,9,9


Primary Δ oracle blind-spot rescue @30: 1.863 п.п.
Decision: limited_residual_model_reserve
Внимание: oracle-any — upper bound; его U_c не равен общей fixed capacity K.


In [6]:
artifact_identities = {
    'stage3_oof_predictions': {'path': str(STAGE3_OOF.relative_to(ROOT)), 'sha256': sha256_file(STAGE3_OOF)},
    'stage3_error_analysis_results': {'path': str(STAGE3_RESULTS.relative_to(ROOT)), 'sha256': sha256_file(STAGE3_RESULTS)},
    'stage6_tabm_oof_v4': {'path': str(STAGE6_OOF.relative_to(ROOT)), 'sha256': sha256_file(STAGE6_OOF)},
    'stage6_tabm_results_v4': {'path': str(STAGE6_RESULTS.relative_to(ROOT)), 'sha256': sha256_file(STAGE6_RESULTS)},
    'stage7_tabm_stacking_oof': {'path': str(STAGE7_OOF.relative_to(ROOT)), 'sha256': sha256_file(STAGE7_OOF)},
    'stage8_ft_transformer_oof': {'path': str(STAGE8_OOF.relative_to(ROOT)), 'sha256': sha256_file(STAGE8_OOF)},
    'stage9_rank_capacity_results': {'path': str(STAGE9_RESULTS.relative_to(ROOT)), 'sha256': sha256_file(STAGE9_RESULTS)},
}
result = {
    'experiment': 'Stage 10',
    'version': 'V1',
    'status': 'completed',
    'diagnostic_only': True,
    'final_test_used': False,
    'oracle_any_scope': 'upper_bound_diagnostic_only; not a shared fixed-capacity production system',
    'artifact_identities': artifact_identities,
    'preflight': preflight,
    'ranking': {'probability_order': 'DESC', 'exact_tie_break': 'working_index ASC', 'capacities': CAPACITIES},
    'locked_score_set': list(LOCKED_SCORE_NAMES),
    'blind_spot': blind_spot_check,
    'overall_default_count': default_count,
    'capacities': capacity_records,
    'primary_delta_oracle_blind_spot_rescue_30_pp': primary_delta_oracle_blind_spot_rescue_30_pp,
    'decision': decision,
    'next_step_owner': 'Technical Coordinator',
    'limitations': [
        'oracle-any — только upper-bound diagnostic: U_c может быть больше K, поэтому это не система с общей capacity 30%.',
        'OOF random-CV не доказывает temporal stability.',
        'Ни retraining, ни новые prediction, ни final test не выполнялись.',
    ],
}
summary = {
    key: result[key] for key in (
        'experiment', 'version', 'status', 'diagnostic_only', 'final_test_used', 'oracle_any_scope',
        'preflight', 'ranking', 'locked_score_set', 'blind_spot', 'overall_default_count', 'capacities',
        'primary_delta_oracle_blind_spot_rescue_30_pp', 'decision', 'next_step_owner', 'limitations',
    )
}

generated_path = GENERATED / 'stage10_residual_model_reserve_results_V1.json'
summary_path = SUMMARY / 'stage10_residual_model_reserve_summary_V1.json'
generated_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Сохранён: {generated_path.relative_to(ROOT)}')
print(f'Сохранён: {summary_path.relative_to(ROOT)}')

Сохранён: reports\generated\stage10_residual_model_reserve_results_V1.json
Сохранён: reports\summary\stage10_residual_model_reserve_summary_V1.json


# Результат исследования

## ФАКТЫ

Stage 10 V1 завершён успешно.

Это диагностический анализ уже сохранённых OOF predictions. Новое обучение моделей и новые predictions не выполнялись.

Перед расчётом была проверена совместимость используемых artifacts:

- рабочая выборка содержит `289 614` наблюдений;
- target и working-index identity между используемыми Stage совпадают;
- для Stage 6 V4 отдельно подтверждена совместимость сохранённого порядка и target;
- все семь используемых score series содержат конечные значения;
- зафиксированная Stage 3 blind spot сохранена без переопределения и содержит `805` дефолтных наблюдений;
- final test не использовался.

В анализ вошли семь сохранённых OOF ranking series:

- CatBoost Stage 3;
- XGBoost Stage 3;
- LightGBM Stage 3;
- `GBDT_mean` Stage 7;
- TabM standalone Stage 6 V4;
- TabM stacking Stage 7;
- FT-Transformer Stage 8.

Для каждой модели отдельно выбирался её top-K при capacity `10%`, `20%` и `30%`.

После этого строился диагностический `oracle-any`: объект считался попавшим в oracle high-risk, если он оказался в top-K хотя бы у одной из этих моделей.

### Результат на Stage 3 blind spot

| Исходная capacity каждой модели | K | Фактический размер oracle union | Фактическая oracle capacity | GBDT_mean rescue | Oracle-any rescue |
|---:|---:|---:|---:|---:|---:|
| 10% | 28 961 | 39 850 | 13.760% | 0 / 805 | 0 / 805 |
| 20% | 57 922 | 74 603 | 25.759% | 0 / 805 | 0 / 805 |
| 30% | 86 884 | 108 450 | 37.446% | 0 / 805 | 15 / 805 (1.863%) |

При основной capacity `30%`:

- `GBDT_mean` не поднимает в high-risk ни одного из `805` blind-spot дефолтов;
- oracle union поднимает `15 из 805`;
- прирост oracle blind-spot rescue относительно `GBDT_mean` составляет **+1.863 п.п.**

Заранее зафиксированный критерий наличия material residual model reserve:

**не менее +5.0 п.п.**

Фактическое значение ниже этого порога.

**Decision: `limited_residual_model_reserve`.**

### Какие модели дали эти 15 дополнительных случаев

При capacity `30%`:

| Альтернативная модель | Дополнительные blind-spot дефолты |
|---|---:|
| CatBoost Stage 3 | 0 |
| XGBoost Stage 3 | 0 |
| LightGBM Stage 3 | 0 |
| TabM standalone Stage 6 V4 | 6 |
| TabM stacking Stage 7 | 0 |
| FT-Transformer Stage 8 | 9 |

Итого альтернативные модели добавляют `15` blind-spot случаев, которые `GBDT_mean` не поднял в свой top-30%.

## ИНТЕРПРЕТАЦИЯ

В уже исследованных моделях действительно существует небольшой дополнительный ranking signal относительно `GBDT_mean`.

Это видно по 15 трудным дефолтам, которые `GBDT_mean` не поднял в high-risk, но которые оказались достаточно высоко в ranking других моделей.

Однако масштаб этого сигнала мал.

Из всей фиксированной blind spot, содержащей `805` дефолтов, даже диагностический oracle смог дополнительно поднять только:

**15 объектов, или 1.863%.**

Это существенно ниже заранее установленного порога `5 п.п.`, поэтому наличие material residual model reserve **не подтверждено**.

Особенно показательно, что при capacity `10%` и `20%` ни одна из исследованных моделей вообще не спасает ни одного объекта этой blind spot. Дополнительные случаи появляются только на уровне 30%.

Основной вклад в эти 15 случаев дают:

- standalone TabM — `6`;
- FT-Transformer — `9`.

Остальные исследованные модели новых случаев здесь не добавляют.

Таким образом, предыдущие отрицательные результаты TabM и FT-Transformer не означают, что их ranking полностью идентичен `GBDT_mean`. Небольшие уникальные сигналы у них есть.

Но Stage 10 показывает, что **их совокупный масштаб недостаточен, чтобы считать model diversity среди уже проверенных подходов крупным неиспользованным резервом**.

Это усиливает evidence в пользу того, что ограничения текущего результата могут находиться не только в выборе архитектуры модели, но и в доступной информации 47-признакового пространства.

При этом эксперимент не доказывает, что 47 признаков принципиально исчерпаны или что никакая другая модель не сможет получить из них дополнительный сигнал.

## ВАЖНО ПРО ORACLE

Рост общего default capture у `oracle-any` нельзя напрямую сравнивать с `GBDT_mean` как преимущество модели.

Например, при исходной capacity `30%`:

- `GBDT_mean` действительно использует top-30%;
- oracle union фактически содержит `108 450` строк, то есть около `37.45%` рабочей выборки.

Поэтому `oracle-any` получает дополнительную вместимость.

Это сделано намеренно: oracle здесь является **оптимистичной диагностической верхней границей**, а не моделью или системой с общей fixed capacity 30%.

Даже в таких благоприятных условиях он спасает только `15 из 805` blind-spot дефолтов.

## ОГРАНИЧЕНИЯ

- `oracle-any` не является обученной моделью или deployable ensemble. Для каждого объекта он фактически пользуется знанием того, что достаточно успеха хотя бы одной модели.
- Oracle union имеет большую фактическую capacity, чем отдельная модель, поэтому его общий default capture нельзя трактовать как честное same-capacity превосходство над `GBDT_mean`.
- Результат относится только к уже исследованному набору моделей. Он не доказывает отсутствие полезного сигнала у всех возможных будущих алгоритмов.
- Анализ основан на OOF predictions random CV и не доказывает temporal stability.
- Experiment не доказывает business benefit и не определяет рабочую risk capacity или threshold.
- Результат усиливает гипотезу об information / feature limitation, но сам по себе не доказывает, что текущие 47 признаков полностью исчерпали доступный прогностический сигнал.
- Ни retraining, ни новые predictions, ни final test не выполнялись.

## ВЫВОД

Исследовательский вопрос Stage 10 закрыт:

**среди уже проверенных моделей существует небольшой остаточный ranking signal поверх `GBDT_mean`, но material residual model reserve не обнаружен.**

Даже оптимистичный oracle union при основном уровне 30% дополнительно поднимает только `15 из 805` фиксированных blind-spot дефолтов:

**+1.863 п.п. против заранее зафиксированного порога +5.0 п.п.**

Поэтому текущие результаты не дают достаточного основания начинать новый stacking или blending только ради объединения уже существующих моделей.



## СЛЕДУЮЩИЙ ШАГ

После Stage 10 основную ветку поиска новых моделей рационально остановить.

За Stage 1–10 уже сложилась последовательная исследовательская картина:

- без `Q_B1_norm` и `Q_B2_norm` построен сильный разрешённый baseline;
- Stage 3 выявил общую blind spot из `805` трудных дефолтов;
- Stage 5 показал наличие недостающего диагностического сигнала, который текущие 47 разрешённых признаков воспроизводят не полностью;
- TabM standalone не улучшил baseline;
- TabM stacking не дал material benefit;
- FT-Transformer также не дал material benefit;
- более высокий Recall FT-Transformer при threshold `0.5` не подтвердился как существенная rank complementarity;
- даже оптимистичный oracle-анализ всех уже сохранённых моделей дополнительно спас только `15 из 805` blind-spot дефолтов (`+1.863 п.п.`), причём фактическая union capacity выросла до `37.446%`.

В совокупности эти результаты дают основание считать, что **текущее ограничение скорее связано с информацией, содержащейся в доступном 47-признаковом представлении, чем с отсутствием ещё одной подходящей архитектуры модели**.

Это исследовательская интерпретация накопленного evidence, а не доказательство того, что 47 признаков принципиально исчерпаны или что никакая другая модель не сможет дать улучшение.

### Что делаем дальше

Следующий основной этап — **собрать результаты Stage 1–10 в единую защищаемую исследовательскую историю**:

**baseline → общие ошибки → missing signal → современные модели → stacking → complementarity → oracle upper bound → вывод о текущем ограничении 47-feature representation.**

Параллельно следует дождаться независимого эксперимента RealMLP.

Его результат рассматривается как дополнительная независимая проверка:

- если RealMLP также не покажет material gain, это усилит уже полученный вывод;
- если RealMLP неожиданно даст material gain, появится новое evidence и основание вернуться к вопросу о model-family ceiling.

До появления такого нового evidence очередной дорогой model search не требуется.

### Когда model research стоит возобновить

Вернуться к основной модельной ветке имеет смысл при появлении хотя бы одного из следующих оснований:

1. RealMLP показывает material gain;
2. появляется новый временно корректный источник признаков или надёжная row-level временная привязка;
3. заказчик задаёт конкретный business operating point — допустимую review capacity, FP burden или другой policy constraint, который создаёт новый содержательный вопрос по Recall / Precision / capacity.

Без одного из этих триггеров следующая модель, скорее всего, увеличит число алгоритмов в сравнительной таблице, но не изменит исследовательский вывод.

**Следующий практический шаг: research synthesis Stage 1–10 и подготовка evidence package для презентации и защиты.**

Final test по-прежнему не используется для выбора направления исследования.